# RVC Voice Cloning - Fine Tuning en Colab
Entrena la red neuronal RVC con tu audio. Al final descarga el modelo y usalo en VoiceMod.

In [ ]:
# CELDA 1: Instalar + descargar TODO
import os, gc, urllib.request

print('Instalando dependencias...')
!pip install -q torch torchaudio pyworld librosa transformers

print('Creando carpetas...')
os.makedirs('models/base', exist_ok=True)
os.makedirs('core/rvc_model', exist_ok=True)

print('Descargando modelo RVC base (73 MB)...')
!wget -q -O models/base/f0G40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth

print('Descargando arquitectura RVC...')
for f in ['models.py','modules.py','commons.py','attentions.py','transforms.py']:
    url = f'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/infer_pack/{f}'
    urllib.request.urlretrieve(url, f'core/rvc_model/{f}')

print('Parcheando imports y hardcodes...')
for f in ['models.py','modules.py','commons.py','attentions.py','transforms.py']:
    with open(f'core/rvc_model/{f}') as fp: c = fp.read()
    c = c.replace('from infer_pack', 'from core.rvc_model')
    c = c.replace('nn.Linear(256, hidden_channels)', 'nn.Linear(in_dim, hidden_channels)')
    c = c.replace('gin_channels=0,', 'gin_channels=gin_channels,')
    c = c.replace('nn.Linear(self.spk_embed_dim, gin_channels)', 'nn.Embedding(self.spk_embed_dim, gin_channels)')
    with open(f'core/rvc_model/{f}','w') as fp: fp.write(c)

print('Agregando metodo forward al modelo...')
with open('core/rvc_model/models.py') as fp: code = fp.read()
old = 'return o, x_mask, (z, z_p, m_p, logs_p)\nclass SynthesizerTrn256NSFkm'
new = '''return o, x_mask, (z, z_p, m_p, logs_p)

    def forward(self, phone, phone_lengths, pitch, pitchf, y, y_lengths, ds):
        m_p, logs_p, x_mask = self.enc_p(phone, pitch, phone_lengths)
        z, m_q, logs_q, y_mask = self.enc_q(y, y_lengths, g=None)
        z_p = self.flow(z, y_mask, g=None)
        z_slice, ids_slice = commons.rand_slice_segments(z, y_lengths, self.segment_size)
        pitchf = commons.slice_segments2(pitchf, ids_slice, self.segment_size)
        o = self.dec(z_slice, pitchf, g=None)
        return o, ids_slice, x_mask, y_mask, (z, z_p, m_p, logs_p, m_q, logs_q)

class SynthesizerTrn256NSFkm'''
code = code.replace(old, new)
with open('core/rvc_model/models.py','w') as fp: fp.write(code)

print('LISTO - Ahora ejecuta la siguiente celda para subir tu audio')


### Subi tu audio aqui
Arrastra el archivo .m4a, .wav o .mp3 de la persona a clonar (max 2 minutos recomendado)

In [ ]:
# CELDA 2: Subir audio
from google.colab import files
print('Selecciona tu archivo de audio...')
uploaded = files.upload()
AUDIO_PATH = list(uploaded.keys())[0]
print(f'Audio cargado: {AUDIO_PATH}')


In [ ]:
# CELDA 3: Extraer features del audio
import torch, numpy as np, librosa, pyworld as pw, gc
torch.set_num_threads(2)

print('Cargando audio...')
audio, sr = librosa.load(AUDIO_PATH, sr=40000, mono=True)
print(f'Duracion: {len(audio)/sr:.0f}s')
audio = audio[:90*sr]
audio = audio/(np.max(np.abs(audio))+1e-8)

print('Cargando HuBERT para extraer features...')
from transformers import HubertModel, Wav2Vec2FeatureExtractor
hubert_fe = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
hubert = HubertModel.from_pretrained('facebook/hubert-base-ls960')
hubert.eval()

all_feats, all_f0, all_f0f, all_mel = [], [], [], []
CHUNK = sr*5
print('Extrayendo features (5s por chunk)...')
for start in range(0, len(audio), CHUNK):
    end = min(start+CHUNK, len(audio))
    seg = audio[start:end].astype(np.float64)
    if len(seg) < sr: break
    
    audio16 = librosa.resample(seg, orig_sr=40000, target_sr=16000)
    inputs = hubert_fe(audio16.astype(np.float32), sampling_rate=16000, return_tensors='pt', padding=True)
    with torch.no_grad():
        feats = hubert(**inputs).last_hidden_state.cpu().numpy()
    
    f0, t = pw.dio(seg, sr, f0_floor=50, f0_ceil=1100)
    f0 = pw.stonemask(seg, f0, t, sr)
    S = librosa.stft(seg, n_fft=2048, hop_length=400)
    mel = np.abs(S)
    
    f0_int = np.clip((f0/1100*256).astype(np.int64), 0, 255)
    min_t = min(len(f0_int), feats.shape[1], mel.shape[1])
    all_feats.append(torch.from_numpy(feats[:,:min_t,:]).float())
    all_f0.append(torch.from_numpy(f0_int[:min_t]).long())
    all_f0f.append(torch.from_numpy(np.where(f0[:min_t]>0,f0[:min_t],0).astype(np.float32)))
    all_mel.append(torch.from_numpy(mel[:,:min_t]).unsqueeze(0).float())
    print(f'  {start//sr}-{end//sr}s')

feats_all = torch.cat(all_feats, dim=1)
f0_all = torch.cat(all_f0).unsqueeze(0)
f0f_all = torch.cat(all_f0f).unsqueeze(0)
mel_all = torch.cat(all_mel, dim=2)

del hubert, hubert_fe; gc.collect()
print(f'Features listos: {feats_all.shape} {f0_all.shape} {mel_all.shape}')


In [ ]:
# CELDA 4: Cargar modelo y ENTRENAR (GPU)
import sys; sys.path.insert(0, '.')
import torch, numpy as np, gc
from core.rvc_model.models import SynthesizerTrnMs256NSF

print('Cargando modelo RVC base...')
ckpt = torch.load('models/base/f0G40k.pth', map_location='cuda')
model = SynthesizerTrnMs256NSF(
    spec_channels=1025, segment_size=12800, inter_channels=192,
    hidden_channels=192, filter_channels=768, n_heads=2, n_layers=6,
    kernel_size=3, p_dropout=0, resblock='1',
    resblock_kernel_sizes=[3,7,11],
    resblock_dilation_sizes=[[1,3,5],[1,3,5],[1,3,5]],
    upsample_rates=[10,10,2,2], upsample_initial_channel=512,
    upsample_kernel_sizes=[16,16,4,4],
    spk_embed_dim=109, gin_channels=256,
    sr=40000, is_half=True, phone_dim=768,
).cuda()
model.load_state_dict(ckpt['model'], strict=False)
model.train()
del ckpt; gc.collect()
print('Modelo listo')

EPOCHS = 30
TRAIN_FRAMES = 40000//400*3
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

print(f'Entrenando {EPOCHS} epocas en GPU...')
for ep in range(1, EPOCHS+1):
    t = feats_all.size(1)
    s = np.random.randint(0, max(1, t-TRAIN_FRAMES))
    e = s+TRAIN_FRAMES
    
    fb = feats_all[:, s:e, :].cuda()
    f0b = f0_all[:, s:e].cuda()
    ffb = f0f_all[:, s:e].cuda()
    mb = mel_all[:, :1025, s:e].cuda()
    
    opt.zero_grad()
    o, _, _, _, _ = model(
        fb.half(), torch.tensor([fb.size(1)]),
        f0b, ffb.half(), mb.half(),
        torch.tensor([mb.size(2)]), torch.tensor([0]).cuda()
    )
    loss = o.abs().mean()
    loss.backward()
    opt.step()
    
    if ep==1 or ep%5==0:
        print(f'  Epoca {ep}/{EPOCHS}: loss={loss.item():.6f}')
        torch.save({'model': {k:v.cpu() for k,v in model.state_dict().items()}}, 'voz_jimmy.pth')

torch.save({'model': {k:v.cpu() for k,v in model.state_dict().items()}}, 'voz_jimmy.pth')
print('ENTRENAMIENTO COMPLETADO!')


In [ ]:
# CELDA 5: Descargar modelo entrenado
from google.colab import files
files.download('voz_jimmy.pth')
print('Descarga completa. Copia voz_jimmy.pth a la carpeta models/trained/ del proyecto')
